## Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import ast

## Data Preparation
Concatenated dataset from external source into one csv file, changing columns datatype and renaming columns.

In [2]:
df = pd.read_json(r"E:\End-to-End-MLOps-MWorld-Champions\data\link\stage.json")
df

,0,1,2,3
0,B-Tier_Tournaments,M Challenge Cup Mekong Season 6,Group Stage,https://liquipedia.net/mobilelegends/M_Challen...
1,B-Tier_Tournaments,M Challenge Cup Mekong Season 6,Playoffs,https://liquipedia.net/mobilelegends/M_Challen...
2,B-Tier_Tournaments,ESN National Championship 2025,Knockout Stage,https://liquipedia.net/mobilelegends/ESN/Natio...
3,B-Tier_Tournaments,Vietnam MLBB Championship 2025 Winter,Group Stage,https://liquipedia.net/mobilelegends/Vietnam_M...
4,B-Tier_Tournaments,Vietnam MLBB Championship 2025 Winter,Playoffs,https://liquipedia.net/mobilelegends/Vietnam_M...
...,...,...,...,...
63,A-Tier_Tournaments,MTC Turkiye Championship Season 5,Playoffs,https://liquipedia.net/mobilelegends/MTC_Turki...
64,A-Tier_Tournaments,MPL Cambodia Season 8,Playoffs,https://liquipedia.net/mobilelegends/MPL/Cambo...
65,A-Tier_Tournaments,MPL Cambodia Season 8,Regular Season,https://liquipedia.net/mobilelegends/MPL/Cambo...
66,A-Tier_Tournaments,ESL Snapdragon Pro Series 2025 Masters,Group Stage,https://liquipedia.net/mobilelegends/ESL/Snapd...


In [3]:
file_path = []
for root, dirs, files in os.walk(r"E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat", topdown=False):
    for file_name in sorted(files):
        path = os.path.join(root, file_name)
        file_path.append(path)

file_path

['E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - ESL_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - GOF_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - ID_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - Japan_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - KH_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - LATAM_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - MCCM_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - MCC_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap - MCS_team_stat.csv',
 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\team stat\\match recap

In [4]:
df = []
for file in file_path:
    data = pd.read_csv(file)
    if data.empty:
        continue

    print(f"Processing file : {file}.")
    data = data.drop(labels=["ID"], axis=1)
    data = data.dropna(subset=["date"])
    df.append(data)

Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - ESL_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - GOF_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - ID_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - Japan_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - KH_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - LATAM_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - MCCM_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - MCC_team_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\team stat\match recap - MCS_team

In [5]:
df = pd.concat(df, ignore_index=True)
df.columns.to_list()

['date',
 'tournament',
 'stage',
 'bracket',
 'team',
 'duration',
 'side',
 'kill',
 'death',
 'assist',
 'tower',
 'lord',
 'turtle',
 'gold',
 'gold_lead_5',
 'gold_lead_10',
 'gold_lead_15',
 'gold_lead_20']

In [6]:
df = df.rename(columns={
    "tournament": "tournament_name",
    "stage": "stage_name",
    "bracket": "bracket_name",
    "player": "player_name",
    "team": "team_name",
    "side": "team_side",
    "role ": "player_role",
    "kill": "kills",
    "death": "deaths",
    "assist": "assists",
    "tower": "towers",
    "lord": "lords",
    "turtle": "turtles",
    "gold": "gold_total",
    "gold_lead_5": "gold_5mnt",
    "gold_lead_10": "gold_10mnt",
    "gold_lead_15": "gold_15mnt",
    "gold_lead_20": "gold_20mnt"
    })
bulan_map = {
    "Januari": "January",
    "Februari": "February",
    "Maret": "March",
    "April": "April",
    "Mei": "May",
    "Juni": "June",
    "Juli": "July",
    "Agustus": "August",
    "September": "September",
    "Oktober": "October",
    "November": "November",
    "Desember": "December"
}
df["date"] = [datetime.strptime(str(i), "%d %B %Y").date() for i in df["date"].replace(bulan_map, regex=True)]
df

,date,tournament_name,stage_name,bracket_name,team_name,duration,team_side,kills,deaths,assists,towers,lords,turtles,gold_total,gold_5mnt,gold_10mnt,gold_15mnt,gold_20mnt
0,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,Bigetron by Vitality,"24,12",Blue,20.0,18.0,42.0,8.0,2.0,3.0,75300.0,-500.0,1300.0,-600.0,2100.0
1,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,Team Liquid ID,"24,12",Red,18.0,20.0,35.0,7.0,2.0,0.0,68700.0,500.0,-1300.0,600.0,-2100.0
2,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,Bigetron by Vitality,"21,48",Red,11.0,18.0,27.0,1.0,0.0,1.0,60000.0,-200.0,-5300.0,-6800.0,-3400.0
3,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,Team Liquid ID,"21,48",Blue,18.0,11.0,46.0,9.0,4.0,2.0,63200.0,200.0,5300.0,6800.0,3400.0
4,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,Bigetron by Vitality,"18,55",Blue,5.0,19.0,11.0,2.0,0.0,1.0,51100.0,-3600.0,-3900.0,-7500.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6741,2025-10-19,Vietnam MLBB Championship 2025 Winter,Playoffs,Grand Final,Saigon Phantom,"12,98",Red,6.0,13.0,8.0,0.0,0.0,0.0,35000.0,-1600.0,-9100.0,NaN,NaN
6742,2025-10-19,Vietnam MLBB Championship 2025 Winter,Playoffs,Grand Final,RLG SE,"29,40",Red,15.0,17.0,33.0,6.0,1.0,3.0,85400.0,1200.0,3300.0,-1600.0,-2400.0
6743,2025-10-19,Vietnam MLBB Championship 2025 Winter,Playoffs,Grand Final,Saigon Phantom,"29,40",Blue,9.0,15.0,40.0,7.0,4.0,0.0,82200.0,-1200.0,-3300.0,1600.0,2400.0
6744,2025-10-19,Vietnam MLBB Championship 2025 Winter,Playoffs,Grand Final,RLG SE,"16,78",Red,6.0,5.0,18.0,9.0,3.0,3.0,53200.0,-500.0,5800.0,9200.0,NaN


In [7]:
df.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\external\team_stats.csv", index=False)

In [8]:
file_path = []
for root, dirs, files in os.walk(r"E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat", topdown=False):
    for file_name in sorted(files):
        path = os.path.join(root, file_name)
        file_path.append(path)

print(file_path)

['E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - ESL_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - GOF_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - ID_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - Japan_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - KH_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - LATAM_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - MCCM_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - MCC_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\external\\player stat\\match recap - MCS_player_stat.csv', 'E:\\End-to-End-MLOps-MWorld-Champions\\data\\exter

In [9]:
df1 = []
for file in file_path:
    data = pd.read_csv(file)
    if data.empty:
        continue

    print(f"Processing file : {file}.")
    data = data.dropna(axis=0)
    df1.append(data)

Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - ESL_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - GOF_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - ID_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - Japan_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - KH_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - LATAM_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - MCCM_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\player stat\match recap - MCC_player_stat.csv.
Processing file : E:\End-to-End-MLOps-MWorld-Champions\data\external\

In [10]:
df = pd.concat(df1, ignore_index=True)
df.columns.to_list()

['tournament',
 'stage',
 'player',
 'team',
 'role',
 'kill',
 'death',
 'assist',
 'hero']

In [11]:
df = df.rename(columns={
    "tournament": "tournament_name",
    "stage": "stage_name",
    "player": "player_name",
    "team": "team_name",
    "role": "player_role",
    "kill": "kills",
    "death": "deaths",
    "assist": "assists",
    "hero": 'hero_name'
    }).sort_values(by=["player_role", "team_name"], ascending=True, ignore_index=True)
df

,tournament_name,stage_name,player_name,team_name,player_role,kills,deaths,assists,hero_name
0,MSC 2025 Mongolia,Qualifier,Corazon,1 Team,EXP Lane,1.0,3.0,3.0,Hylos
1,MSC 2025 Mongolia,Qualifier,Corazon,1 Team,EXP Lane,1.0,2.0,4.0,Hilda
2,MSC 2025 Mongolia,Qualifier,Corazon,1 Team,EXP Lane,0.0,2.0,0.0,Gloo
3,MSC 2025 Mongolia,Qualifier,Corazon,1 Team,EXP Lane,3.0,3.0,8.0,Edith
4,MSC 2025 Mongolia,Qualifier,Corazon,1 Team,EXP Lane,0.0,2.0,1.0,Hylos
...,...,...,...,...,...,...,...,...,...
45260,MLBB Super League Myanmar Season 1,Regular Season,Light,Zino Esports,Roamer,1.0,3.0,12.0,Badang
45261,MLBB Super League Myanmar Season 1,Play-Ins,Light,Zino Esports,Roamer,3.0,1.0,7.0,Khaleed
45262,MLBB Super League Myanmar Season 1,Play-Ins,Light,Zino Esports,Roamer,1.0,7.0,10.0,Baxia
45263,MLBB Super League Myanmar Season 1,Play-Ins,Light,Zino Esports,Roamer,1.0,4.0,4.0,Badang


In [12]:
df.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\external\player_stats.csv", index=False)

## Data Cleaning

Cleaning the data by merging using artificial keys and changing dataset into the correct datatypes. divided dataset into tournament, matches, games, teams, picks, and bans then stored as csv.

In [40]:
teams_stat_raw = pd.read_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\external\team_stats.csv").reset_index(names="id")
teams_stat_raw["team_name"] = teams_stat_raw["team_name"].str.lower()
teams_stat_raw[["kills", "assists", "deaths", "towers", "lords", "turtles", "gold_total", "gold_5mnt", "gold_10mnt", "gold_15mnt", "gold_20mnt"]] = teams_stat_raw[["kills", "assists", "deaths", "towers", "lords", "turtles", "gold_total", "gold_5mnt", "gold_10mnt", "gold_15mnt", "gold_20mnt"]].astype("Int64")
teams_stat_raw.head(10)

,id,date,tournament_name,stage_name,bracket_name,team_name,duration,team_side,kills,deaths,assists,towers,lords,turtles,gold_total,gold_5mnt,gold_10mnt,gold_15mnt,gold_20mnt
0,0,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,bigetron by vitality,"24,12",Blue,20,18,42,8,2,3,75300,-500,1300,-600,2100
1,1,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,team liquid id,"24,12",Red,18,20,35,7,2,0,68700,500,-1300,600,-2100
2,2,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,bigetron by vitality,"21,48",Red,11,18,27,1,0,1,60000,-200,-5300,-6800,-3400
3,3,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,team liquid id,"21,48",Blue,18,11,46,9,4,2,63200,200,5300,6800,3400
4,4,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,bigetron by vitality,"18,55",Blue,5,19,11,2,0,1,51100,-3600,-3900,-7500,<NA>
5,5,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,team liquid id,"18,55",Red,19,5,39,9,3,2,58300,3600,3900,7500,<NA>
6,6,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,onic philippines,"11,38",Red,11,4,27,9,1,2,40300,200,7800,<NA>,<NA>
7,7,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,ybingame,"11,38",Blue,4,11,9,2,0,1,27300,-200,-7800,<NA>,<NA>
8,8,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,onic philippines,"11,45",Blue,22,3,47,9,1,3,43400,4700,12100,<NA>,<NA>
9,9,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,ybingame,"11,45",Red,3,22,5,0,0,0,26500,-4700,-12100,<NA>,<NA>


In [41]:
teams_stat_raw.columns.tolist()

['id',
 'date',
 'tournament_name',
 'stage_name',
 'bracket_name',
 'team_name',
 'duration',
 'team_side',
 'kills',
 'deaths',
 'assists',
 'towers',
 'lords',
 'turtles',
 'gold_total',
 'gold_5mnt',
 'gold_10mnt',
 'gold_15mnt',
 'gold_20mnt']

Change data structure into wide format by divided home and away based on row because data pattern are the same. Then, merge into a dateset use by reset index as the keys.

In [42]:
teams_stat_detail = teams_stat_raw.copy()

home = teams_stat_detail.drop(teams_stat_raw.index[1::2]).reset_index(drop=True)
away = teams_stat_detail.drop(teams_stat_raw.index[0::2]).reset_index(drop=True)

home = home.reset_index(names="keys")
away = away.reset_index(names="keys")

teams_stat_raw_merged = home.merge(away, on="keys")
teams_stat_raw_merged.head(10)

,keys,id_x,date_x,tournament_name_x,stage_name_x,bracket_name_x,team_name_x,duration_x,team_side_x,kills_x,...,deaths_y,assists_y,towers_y,lords_y,turtles_y,gold_total_y,gold_5mnt_y,gold_10mnt_y,gold_15mnt_y,gold_20mnt_y
0,0,0,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,bigetron by vitality,"24,12",Blue,20,...,20,35,7,2,0,68700,500,-1300,600,-2100
1,1,2,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,bigetron by vitality,"21,48",Red,11,...,11,46,9,4,2,63200,200,5300,6800,3400
2,2,4,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,bigetron by vitality,"18,55",Blue,5,...,5,39,9,3,2,58300,3600,3900,7500,<NA>
3,3,6,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,onic philippines,"11,38",Red,11,...,11,9,2,0,1,27300,-200,-7800,<NA>,<NA>
4,4,8,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,onic philippines,"11,45",Blue,22,...,22,5,0,0,0,26500,-4700,-12100,<NA>,<NA>
5,5,10,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,influence rage,"21,35",Blue,11,...,11,56,5,1,2,62500,-500,-900,-1900,-100
6,6,12,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,influence rage,"18,60",Red,17,...,17,32,4,1,0,57500,-600,-6200,-4800,<NA>
7,7,14,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,onic,"22,55",Red,12,...,12,18,6,1,1,62500,-1700,-1200,500,2200
8,8,16,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,onic,"17,88",Blue,7,...,7,27,8,3,2,55200,600,-1800,4600,<NA>
9,9,18,2025-04-07,ESL Snapdragon Pro Series 2025 Masters,Group Stage,Group Stage,onic,"12,80",Red,18,...,18,13,1,0,1,33100,-300,-3800,<NA>,<NA>


In [43]:
teams_stat_raw_merged.columns.tolist()

['keys',
 'id_x',
 'date_x',
 'tournament_name_x',
 'stage_name_x',
 'bracket_name_x',
 'team_name_x',
 'duration_x',
 'team_side_x',
 'kills_x',
 'deaths_x',
 'assists_x',
 'towers_x',
 'lords_x',
 'turtles_x',
 'gold_total_x',
 'gold_5mnt_x',
 'gold_10mnt_x',
 'gold_15mnt_x',
 'gold_20mnt_x',
 'id_y',
 'date_y',
 'tournament_name_y',
 'stage_name_y',
 'bracket_name_y',
 'team_name_y',
 'duration_y',
 'team_side_y',
 'kills_y',
 'deaths_y',
 'assists_y',
 'towers_y',
 'lords_y',
 'turtles_y',
 'gold_total_y',
 'gold_5mnt_y',
 'gold_10mnt_y',
 'gold_15mnt_y',
 'gold_20mnt_y']

In [44]:
teams_stat_raw_merged = teams_stat_raw_merged.drop(columns=["id_x", "id_y", "keys"])

Similar colomns between team_stat dataset and match_raw are they contain date, tournament name, bracket name, team pair and game num. So, create a artificial keys by combine those colomns.

In [45]:
teams_stat_raw_merged["team_pair"] = teams_stat_raw_merged.apply(
    lambda x: sorted([x["team_name_x"], x["team_name_y"]]),
    axis=1
)
teams_stat_raw_merged["match_keys"] = teams_stat_raw_merged[["date_x", "tournament_name_x", "bracket_name_x", "team_pair"]].astype(str).agg("_".join, axis=1)
teams_stat_raw_merged["game_num"] = teams_stat_raw_merged.groupby("match_keys").cumcount() + 1
teams_stat_raw_merged["game_keys"] = teams_stat_raw_merged[["date_x", "tournament_name_x", "game_num", "team_pair"]].astype(str).agg("_".join, axis=1)
teams_stat_raw_merged = teams_stat_raw_merged.sort_values(by=["date_x", "team_name_x"]).reset_index(drop=True)
print(f"Total row: {teams_stat_raw_merged.shape[0]}")
teams_stat_raw_merged.head(10)

Total row: 3373


,date_x,tournament_name_x,stage_name_x,bracket_name_x,team_name_x,duration_x,team_side_x,kills_x,deaths_x,assists_x,...,turtles_y,gold_total_y,gold_5mnt_y,gold_10mnt_y,gold_15mnt_y,gold_20mnt_y,team_pair,match_keys,game_num,game_keys
0,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,onic philippines,"17,87",Blue,21,13,48,...,0,52200,-1800,-4000,-4800,<NA>,"[onic philippines, tnc pro team]",2025-02-28_MPL Philippines Season 15_Group Sta...,1,2025-02-28_MPL Philippines Season 15_1_['onic ...
1,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,onic philippines,"21,92",Red,5,8,8,...,3,60800,67,2700,6400,6000,"[onic philippines, tnc pro team]",2025-02-28_MPL Philippines Season 15_Group Sta...,2,2025-02-28_MPL Philippines Season 15_2_['onic ...
2,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,twisted minds ph,"18,87",Red,7,18,18,...,1,58700,1500,3500,8000,<NA>,"[aurora gaming ph, twisted minds ph]",2025-02-28_MPL Philippines Season 15_Group Sta...,1,2025-02-28_MPL Philippines Season 15_1_['auror...
3,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,twisted minds ph,"13,87",Red,2,9,4,...,3,46000,1400,7800,<NA>,<NA>,"[aurora gaming ph, twisted minds ph]",2025-02-28_MPL Philippines Season 15_Group Sta...,2,2025-02-28_MPL Philippines Season 15_2_['auror...
4,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,aurora gaming ph,"19,80",Blue,9,7,23,...,0,53300,-1600,-6900,-5400,<NA>,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,1,2025-03-01_MPL Philippines Season 15_1_['auror...
5,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,aurora gaming ph,"12,67",Red,4,10,5,...,2,37500,-540,1000,<NA>,<NA>,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,2,2025-03-01_MPL Philippines Season 15_2_['auror...
6,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,aurora gaming ph,"15,98",Blue,22,13,56,...,2,47800,-153,-3500,-2600,<NA>,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,3,2025-03-01_MPL Philippines Season 15_3_['auror...
7,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,team falcons ph,"18,57",Blue,11,15,23,...,1,57800,503,-401,3500,<NA>,"[team falcons ph, team liquid ph]",2025-03-01_MPL Philippines Season 15_Group Sta...,1,2025-03-01_MPL Philippines Season 15_1_['team ...
8,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,team falcons ph,"12,52",Red,5,12,13,...,3,42900,1300,8700,<NA>,<NA>,"[team falcons ph, team liquid ph]",2025-03-01_MPL Philippines Season 15_Group Sta...,2,2025-03-01_MPL Philippines Season 15_2_['team ...
9,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,tnc pro team,"13,20",Blue,18,5,38,...,1,32700,-2200,-8200,<NA>,<NA>,"[ap.bren, tnc pro team]",2025-03-01_MPL Philippines Season 15_Group Sta...,1,2025-03-01_MPL Philippines Season 15_1_['ap.br...


In [46]:
tz_map = {
    "JST": "Asia/Tokyo",          # Japan Standard Time (UTC+9)
    "CST": "Asia/Shanghai",       # China Standard Time (UTC+8) ⚠️ ambiguous
    "GST": "Asia/Dubai",          # Gulf Standard Time (UTC+4)
    "ICT": "Asia/Bangkok",        # Indochina Time (UTC+7)
    "MMT": "Asia/Yangon",         # Myanmar Time (UTC+6:30)
    "MSK": "Europe/Moscow",       # Moscow Time (UTC+3)
    "SGT": "Asia/Singapore",      # Singapore Time (UTC+8)
    "BRT": "America/Sao_Paulo",   # Brasilia Time (UTC-3)
    "TRT": "Europe/Istanbul",     # Turkey Time (UTC+3)
    "AST": "Asia/Riyadh",         # Arabia Standard Time (UTC+3) ⚠️ ambiguous
    "COT": "America/Bogota",      # Colombia Time (UTC-5)
    "ULAT": "Asia/Ulaanbaatar"   # Ulaanbaatar Time (UTC+8)
}

In [47]:
def convert_to_wib(row):
    tz = tz_map.get(row["tz"])
    
    if tz is None:
        return pd.NaT
    
    dt = row["date"].tz_localize(tz)
    return dt.tz_convert("Asia/Jakarta")

Split timezone and date using split method and convert into Asia/Jakarta timezone. same as team_stat dataset, create a artificial keys using similar columns.

In [48]:
match_raw = pd.read_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\raw\match_detail.csv")
match_raw["game_num"] = match_raw["game_num"].astype("Int64")
match_raw["tz"] = [i[-4:].strip() if ":" in i else np.nan for i in match_raw["date"] ]
match_raw["tz"] = match_raw["tz"].bfill()
match_raw["date"] = pd.to_datetime([date[:-4].strip() if ":" in date else np.nan for date in match_raw["date"]])
match_raw["date"] = match_raw["date"].ffill()
match_raw["date"] = match_raw.apply(convert_to_wib, axis=1)
match_raw["date_only"] = [i.date() for i in match_raw["date"]]
match_raw = match_raw.drop(columns=["tz"])
match_raw = match_raw.rename(columns={
    "tournament": "tournament_name",
    "tier": "tier_name",
    "stage": "stage_name",
    "bracket": "bracket_name",
    "map": "map_name"
    }).sort_values(by=["date_only", "tournament_name"], ignore_index=True)
match_raw.loc[match_raw["home_team"].str.lower() == "bigetron esports", "home_team"] = "Bigetron by Vitality"
match_raw.loc[match_raw["away_team"].str.lower() == "bigetron esports", "away_team"] = "Bigetron by Vitality"
match_raw.loc[match_raw["home_team"].str.lower() == "beaiktaa esports", "home_team"] = "Besiktas Esports"
match_raw.loc[match_raw["away_team"].str.lower() == "beaiktaa esports", "away_team"] = "Besiktas Esports"
match_raw.loc[match_raw["home_team"].str.lower() == "rlg vietnam", "home_team"] = "RLG SE"
match_raw.loc[match_raw["away_team"].str.lower() == "rlg vietnam", "away_team"] = "RLG SE"
match_raw["home_team"] = match_raw["home_team"].str.lower()
match_raw["away_team"] = match_raw["away_team"].str.lower()
match_raw["team_pair"] = match_raw.apply(
    lambda x: sorted([x["home_team"], x["away_team"]]),
    axis=1
)
match_raw["match_keys"] = match_raw[["date_only", "tournament_name", "bracket_name", "team_pair"]].astype(str).agg("_".join, axis=1)
match_raw["game_keys"] = match_raw[["date_only", "tournament_name", "game_num", "team_pair"]].astype(str).agg("_".join, axis=1)
match_raw = match_raw.sort_values(by=["date_only", "home_team"]).reset_index(drop=True)
print(f"Total row: {match_raw.shape[0]}")
match_raw.head(10)

Total row: 3373


,date,game_num,home_team,home_alias,away_team,away_alias,home_picks,away_picks,home_bans,away_bans,...,home_status,away_status,tier_name,tournament_name,stage_name,bracket_name,date_only,team_pair,match_keys,game_keys
0,2025-02-28 18:30:00+07:00,1,onic philippines,ONIC,tnc pro team,TNC,"['Cici', 'Joy', 'Pharsa', 'Granger', 'Khaleed']","['Hilda', 'Suyou', 'Luo Yi', 'Moskov', 'Gatotk...","['Yve', 'Phoveus', 'Harith', 'Irithel', 'Badang']","['Zhuxin', 'Fanny', 'Selena', 'Chou', 'Hylos']",...,win,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[onic philippines, tnc pro team]",2025-02-28_MPL Philippines Season 15_Group Sta...,2025-02-28_MPL Philippines Season 15_1_['onic ...
1,2025-02-28 18:30:00+07:00,2,onic philippines,ONIC,tnc pro team,TNC,"['Edith', 'Suyou', 'Cecilion', 'Granger', 'Chou']","['Barats', 'Alpha', 'Aurora', 'Harith', 'Mathi...","['Zhuxin', 'Moskov', 'Phoveus', 'Yve', 'Gloo']","['Gatotkaca', 'Fanny', 'Luo Yi', 'Cici', 'Sele...",...,win,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[onic philippines, tnc pro team]",2025-02-28_MPL Philippines Season 15_Group Sta...,2025-02-28_MPL Philippines Season 15_2_['onic ...
2,2025-02-28 16:00:00+07:00,1,twisted minds ph,TWPH,aurora gaming ph,RORA,"['Hilda', 'Hayabusa', 'Yve', 'Harith', 'Hylos']","['Phoveus', 'Suyou', 'Vexana', 'Granger', 'Chou']","['Fanny', 'Zhuxin', 'Mathilda', 'Selena', 'Jaw...","['Joy', 'Luo Yi', 'Lukas', 'Cici', 'Hanzo']",...,loss,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[aurora gaming ph, twisted minds ph]",2025-02-28_MPL Philippines Season 15_Group Sta...,2025-02-28_MPL Philippines Season 15_1_['auror...
3,2025-02-28 16:00:00+07:00,2,twisted minds ph,TWPH,aurora gaming ph,RORA,"['Gloo', 'Hanzo', 'Faramis', 'Granger', 'Hilda']","['Edith', 'Suyou', 'Pharsa', 'Harith', 'Hylos']","['Mathilda', 'Zhuxin', 'Fanny', 'Valentina', '...","['Joy', 'Luo Yi', 'Lukas', 'Phoveus', 'Cici']",...,loss,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[aurora gaming ph, twisted minds ph]",2025-02-28_MPL Philippines Season 15_Group Sta...,2025-02-28_MPL Philippines Season 15_2_['auror...
4,2025-03-01 18:30:00+07:00,1,aurora gaming ph,RORA,onic philippines,ONIC,"['Badang', 'Suyou', 'Zhuxin', 'Karrie', 'Khale...","['Edith', 'Nolan', 'Mathilda', 'Harith', 'Gato...","['Selena', 'Luo Yi', 'Joy', 'Lukas', 'Cici']","['Fanny', 'Phoveus', 'Alpha', 'Moskov', 'Grang...",...,win,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,2025-03-01_MPL Philippines Season 15_1_['auror...
5,2025-03-01 18:30:00+07:00,2,aurora gaming ph,RORA,onic philippines,ONIC,"['Lukas', 'Suyou', 'Valentina', 'Granger', 'Ch...","['Gatotkaca', 'Joy', 'Faramis', 'Irithel', 'Ch...","['Fanny', 'Zhuxin', 'Selena', 'Luo Yi', 'Yve']","['Mathilda', 'Phoveus', 'Harith', 'Badang', 'V...",...,loss,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,2025-03-01_MPL Philippines Season 15_2_['auror...
6,2025-03-01 18:30:00+07:00,3,aurora gaming ph,RORA,onic philippines,ONIC,"['Edith', 'Alpha', 'Yve', 'Granger', 'Hylos']","['Chou', 'Suyou', 'Pharsa', 'Harith', 'Gatotka...","['Selena', 'Luo Yi', 'Joy', 'Cici', 'Gloo']","['Zhuxin', 'Mathilda', 'Fanny', 'Phoveus', 'Lu...",...,win,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,2025-03-01_MPL Philippines Season 15_3_['auror...
7,2025-03-01 13:30:00+07:00,1,team falcons ph,FLCP,team liquid ph,TLPH,"['Gloo', 'Suyou', 'Pharsa', 'Karrie', 'Khaleed']","['Phoveus', 'Alpha', 'Valentina', 'Granger', '...","['Harith', 'Luo Yi', 'Joy', 'Yve', 'Hanzo']","[

Checking unmatching artifical keys between dataset

In [49]:
set_diff = set(match_raw["game_keys"]).symmetric_difference(set(teams_stat_raw_merged["game_keys"]))
set_diff

set()

Merging team_stat and match_raw into single games based dataset using artificial keys

In [154]:
game_details = teams_stat_raw_merged.merge(match_raw, on="game_keys", how="left")
# game_details_right_only = game_details[game_details["_merge"] == "left_only"]
# game_details_right_only["game_keys"]
game_details.head(10)

,date_x,tournament_name_x,stage_name_x,bracket_name_x,team_name_x,duration_x,team_side_x,kills_x,deaths_x,assists_x,...,away_status,tier_name,tournament_name,stage_name,bracket_name,date_only,team_pair_y,match_keys_y,home_score,away_score
0,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,onic philippines,"17,87",Blue,21,13,48,...,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[onic philippines, tnc pro team]",2025-02-28_MPL Philippines Season 15_Group Sta...,1,0
1,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,onic philippines,"21,92",Red,5,8,8,...,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[onic philippines, tnc pro team]",2025-02-28_MPL Philippines Season 15_Group Sta...,1,0
2,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,twisted minds ph,"18,87",Red,7,18,18,...,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[aurora gaming ph, twisted minds ph]",2025-02-28_MPL Philippines Season 15_Group Sta...,0,1
3,2025-02-28,MPL Philippines Season 15,Regular Season,Group Stage,twisted minds ph,"13,87",Red,2,9,4,...,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-02-28,"[aurora gaming ph, twisted minds ph]",2025-02-28_MPL Philippines Season 15_Group Sta...,0,1
4,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,aurora gaming ph,"19,80",Blue,9,7,23,...,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,1,0
5,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,aurora gaming ph,"12,67",Red,4,10,5,...,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,0,1
6,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,aurora gaming ph,"15,98",Blue,22,13,56,...,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[aurora gaming ph, onic philippines]",2025-03-01_MPL Philippines Season 15_Group Sta...,1,0
7,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,team falcons ph,"18,57",Blue,11,15,23,...,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[team falcons ph, team liquid ph]",2025-03-01_MPL Philippines Season 15_Group Sta...,0,1
8,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,team falcons ph,"12,52",Red,5,12,13,...,win,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[team falcons ph, team liquid ph]",2025-03-01_MPL Philippines Season 15_Group Sta...,0,1
9,2025-03-01,MPL Philippines Season 15,Regular Season,Group Stage,tnc pro team,"13,20",Blue,18,5,38,...,loss,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2025-03-01,"[ap.bren, tnc pro team]",2025-03-01_MPL Philippines Season 15_Group Sta...,1,0


In [155]:
game_details.columns.to_list()

['date_x',
 'tournament_name_x',
 'stage_name_x',
 'bracket_name_x',
 'team_name_x',
 'duration_x',
 'team_side_x',
 'kills_x',
 'deaths_x',
 'assists_x',
 'towers_x',
 'lords_x',
 'turtles_x',
 'gold_total_x',
 'gold_5mnt_x',
 'gold_10mnt_x',
 'gold_15mnt_x',
 'gold_20mnt_x',
 'date_y',
 'tournament_name_y',
 'stage_name_y',
 'bracket_name_y',
 'team_name_y',
 'duration_y',
 'team_side_y',
 'kills_y',
 'deaths_y',
 'assists_y',
 'towers_y',
 'lords_y',
 'turtles_y',
 'gold_total_y',
 'gold_5mnt_y',
 'gold_10mnt_y',
 'gold_15mnt_y',
 'gold_20mnt_y',
 'team_pair_x',
 'match_keys_x',
 'game_num_x',
 'game_keys',
 'date',
 'game_num_y',
 'home_team',
 'home_alias',
 'away_team',
 'away_alias',
 'home_picks',
 'away_picks',
 'home_bans',
 'away_bans',
 'duration',
 'map_name',
 'home_status',
 'away_status',
 'tier_name',
 'tournament_name',
 'stage_name',
 'bracket_name',
 'date_only',
 'team_pair_y',
 'match_keys_y',
 'home_score',
 'away_score']

In [156]:
game_details = game_details.drop(columns=[
    "date_only",
    "date_x",
    "date_y",
    "tournament_name",
    "tournament_name_y",
    "stage_name",
    "stage_name_y",
    "bracket_name",
    "bracket_name_y",
    "duration_x",
    "duration_y",
    "game_num_y",
    "match_keys_y",
    "team_pair_y",
    "team_name_x",
    "team_name_y",
])
game_details = game_details.rename(columns={
    "match_keys_x": "match_keys",
    "tournament_name_x": "tournament_name",
    "stage_name_x": "stage_name",
    "bracket_name_x": "bracket_name",
    "game_num_x": "game_num",
    "team_pair_x": "team_pair",
    "team_side_x": "home_side",
    "team_side_y": "away_side",
    "kills_x": "home_kills",
    "kills_y": "away_kills",
    "deaths_x": "home_deaths",
    "deaths_y": "away_deaths",
    "assists_x": "home_assists",
    "assists_y": "away_assists",
    "towers_x": "home_towers",
    "towers_y": "away_towers",
    "lords_x": "home_lords",
    "lords_y": "away_lords",
    "turtles_x": "home_turtles",
    "turtles_y": "away_turtles",
    "gold_total_x": "home_gold_total",
    "gold_total_y": "away_gold_total",
    "gold_5mnt_x": "home_gold_5mnt",
    "gold_5mnt_y": "away_gold_5mnt",
    "gold_10mnt_x": "home_gold_10mnt",
    "gold_10mnt_y": "away_gold_10mnt",
    "gold_15mnt_x": "home_gold_15mnt",
    "gold_15mnt_y": "away_gold_15mnt",
    "gold_20mnt_x": "home_gold_20mnt",
    "gold_20mnt_y": "away_gold_20mnt"})

In [157]:
game_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3375 entries, 0 to 3374
Data columns (total 47 columns):
 #   Column           Non-Null Count  Dtype                       
---  ------           --------------  -----                       
 0   tournament_name  3375 non-null   object                      
 1   stage_name       3375 non-null   object                      
 2   bracket_name     3375 non-null   object                      
 3   home_side        3363 non-null   object                      
 4   home_kills       3366 non-null   Int64                       
 5   home_deaths      3366 non-null   Int64                       
 6   home_assists     3366 non-null   Int64                       
 7   home_towers      3360 non-null   Int64                       
 8   home_lords       3360 non-null   Int64                       
 9   home_turtles     3360 non-null   Int64                       
 10  home_gold_total  3360 non-null   Int64                       
 11  home_gold_5mnt   

In [158]:
game_details = game_details[[
    "date",
    "match_keys",
    "game_keys",
    "tier_name",
    "tournament_name",
    "stage_name",
    "bracket_name",
    "game_num",
    "map_name",
    "duration",
    "team_pair",
    "home_team",
    "away_team",
    "home_alias",
    "away_alias",
    "home_status",
    "away_status",
    "home_picks",
    "away_picks",
    "home_bans",
    "away_bans",
    "home_side",
    "away_side",
    "home_kills",
    "away_kills",
    "home_assists",
    "away_assists",
    "home_deaths",
    "away_deaths",
    "home_towers",
    "away_towers",
    "home_lords",
    "away_lords",
    "home_turtles",
    "away_turtles",
    "home_gold_total",
    "away_gold_total",
    "home_gold_5mnt",
    "away_gold_5mnt",
    "home_gold_10mnt",
    "away_gold_10mnt",
    "home_gold_15mnt",
    "away_gold_15mnt",
    "home_gold_20mnt",
    "away_gold_20mnt",
    ]]
game_details["game_keys"] = game_details[["date", "tournament_name", "game_num", "team_pair"]].astype(str).agg("_".join, axis=1)
game_details["match_keys"] = game_details[["date", "tournament_name", "bracket_name", "team_pair"]].astype(str).agg("_".join, axis=1)
game_details["game_id"] = game_details.index + 1
game_details.head(10)

,date,match_keys,game_keys,tier_name,tournament_name,stage_name,bracket_name,game_num,map_name,duration,...,away_gold_total,home_gold_5mnt,away_gold_5mnt,home_gold_10mnt,away_gold_10mnt,home_gold_15mnt,away_gold_15mnt,home_gold_20mnt,away_gold_20mnt,game_id
0,2025-02-28 18:30:00+07:00,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,1,default,17:52,...,52200,1800,-1800,4000,-4000,4800,-4800,<NA>,<NA>,1
1,2025-02-28 18:30:00+07:00,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2,default,21:55,...,60800,-67,67,-2700,2700,-6400,6400,-6000,6000,2
2,2025-02-28 16:00:00+07:00,2025-02-28 16:00:00+07:00_MPL Philippines Seas...,2025-02-28 16:00:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,1,default,18:52,...,58700,-1500,1500,-3500,3500,-8000,8000,<NA>,<NA>,3
3,2025-02-28 16:00:00+07:00,2025-02-28 16:00:00+07:00_MPL Philippines Seas...,2025-02-28 16:00:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2,default,13:52,...,46000,-1400,1400,-7800,7800,<NA>,<NA>,<NA>,<NA>,4
4,2025-03-01 18:30:00+07:00,2025-03-01 18:30:00+07:00_MPL Philippines Seas...,2025-03-01 18:30:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,1,default,19:48,...,53300,1600,-1600,6900,-6900,5400,-5400,<NA>,<NA>,5
5,2025-03-01 18:30:00+07:00,2025-03-01 18:30:00+07:00_MPL Philippines Seas...,2025-03-01 18:30:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2,default,12:41,...,37500,540,-540,-1000,1000,<NA>,<NA>,<NA>,<NA>,6
6,2025-03-01 18:30:00+07:00,2025-03-01 18:30:00+07:00_MPL Philippines Seas...,2025-03-01 18:30:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,3,default,16:02,...,47800,153,-153,3500,-3500,2600,-2600,<NA>,<NA>,7
7,2025-03-01 13:30:00+07:00,2025-03-01 13:30:00+07:00_MPL Philippines Seas...,2025-03-01 13:30:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,1,default,18:34,...,57800,-503,503,401,-401,-3500,3500,<NA>,<NA>,8
8,2025-03-01 13:30:00+07:00,2025-03-01 13:30:00+07:00_MPL Philippines Seas...,2025-03-01 13:30:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,2,default,12:31,...,42900,-1300,1300,-8700,8700,<NA>,<NA>,<NA>,<NA>,9
9,2025-03-01 16:00:00+07:00,2025-03-01 16:00:00+07:00_MPL Philippines Seas...,2025-03-01 16:00:00+07:00_MPL Philippines Seas...,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,1,default,13:12,...,32700,2200,-2200,8200,-8200,<NA>,<NA>,<NA>,<NA>,10


Create a match based dataset from game based dataset using pandas aggregation

In [159]:
game_details["home_score"] = np.where(game_details["home_status"] == "win", 1, 0)
game_details["away_score"] = np.where(game_details["away_status"] == "win", 1, 0)

match_result = (
    game_details.groupby("match_keys")
    # .size().sort_values(ascending=False).head(30))
    .agg(
        date=("date","first"),
        tier_name=("tier_name", "first"),
        tournament_name=("tournament_name", "first"),
        stage_name=("stage_name", "first"),
        bracket_name=("bracket_name", "first"),
        team_pair=("team_pair", "first"),
        home_team=("home_team", "first"),
        away_team=("away_team", "first"),
        home_score=("home_score", "sum"),
        away_score=("away_score", "sum")
    )
    .reset_index()
)
match_result["winner"] = match_result.apply(
    lambda x: x["home_team"] if x["home_score"] > x["away_score"] else x["away_team"] if x["home_score"] < x["away_score"] else "tie",
    axis=1
)
match_result["score"] = match_result[["home_score", "away_score"]].astype(str).agg("-".join, axis=1)
match_result["match_keys"] = match_result[["date", "tournament_name", "bracket_name", "team_pair"]].astype(str).agg("_".join, axis=1)
match_result = match_result.sort_values(by=["date", "home_team"]).reset_index(drop=True)
match_result["match_id"] = match_result.index + 1
match_result.head(10)

,match_keys,date,tier_name,tournament_name,stage_name,bracket_name,team_pair,home_team,away_team,home_score,away_score,winner,score,match_id
0,2025-02-28 16:00:00+07:00_MPL Philippines Seas...,2025-02-28 16:00:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[aurora gaming ph, twisted minds ph]",twisted minds ph,aurora gaming ph,0,2,aurora gaming ph,0-2,1
1,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,2025-02-28 18:30:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[onic philippines, tnc pro team]",onic philippines,tnc pro team,2,0,onic philippines,2-0,2
2,2025-03-01 13:30:00+07:00_MPL Philippines Seas...,2025-03-01 13:30:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[team falcons ph, team liquid ph]",team falcons ph,team liquid ph,0,2,team liquid ph,0-2,3
3,2025-03-01 16:00:00+07:00_MPL Philippines Seas...,2025-03-01 16:00:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[ap.bren, tnc pro team]",tnc pro team,ap.bren,2,0,tnc pro team,2-0,4
4,2025-03-01 18:30:00+07:00_MPL Philippines Seas...,2025-03-01 18:30:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[aurora gaming ph, onic philippines]",aurora gaming ph,onic philippines,2,1,aurora gaming ph,2-1,5
5,2025-03-02 16:00:00+07:00_MPL Philippines Seas...,2025-03-02 16:00:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[omega esports, team falcons ph]",omega esports,team falcons ph,0,2,team falcons ph,0-2,6
6,2025-03-02 18:30:00+07:00_MPL Philippines Seas...,2025-03-02 18:30:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[team liquid ph, twisted minds ph]",team liquid ph,twisted minds ph,2,0,team liquid ph,2-0,7
7,2025-03-07 15:15:00+07:00_MPL Indonesia Season...,2025-03-07 15:15:00+07:00,A-Tier_Tournaments,MPL Indonesia Season 15,Regular Season,Group Stage,"[natus vincere, rrq hoshi]",natus vincere,rrq hoshi,0,2,rrq hoshi,0-2,8
8,2025-03-07 16:00:00+07:00_MPL Philippines Seas...,2025-03-07 16:00:00+07:00,A-Tier_Tournaments,MPL Philippines Season 15,Regular Season,Group Stage,"[onic philippines, team falcons ph]",team falcons ph,onic philippines,2,1,team falcons ph,2-1,9
9,2025-03-07 18:15:00+07:00_MPL Indonesia Season...,2025-03-07 18:15:00+07:00,A-Tier_Tournaments,MPL Indonesia Season 15,Regular Season,Group Stage,"[evos, team liquid id]",evos,team liquid id,2,1,evos,2-1,10


In [160]:
# update next exploration
# players_stat_raw = pd.read_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\external\player_stats.csv").sort_values(by=["tournament_name", "player_role"])
# players_stat_raw = players_stat_raw[["player_name", "team_name", "player_role"]].drop_duplicates().sort_values(by="player_name").reset_index(drop=True)
# players_stat_raw["player_name"] = players_stat_raw["player_name"].str.lower()
# players_stat_raw["team_name"] = players_stat_raw["team_name"].str.lower()
# players_stat_raw

In [161]:
# update next exploration
# players_raw = pd.read_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\raw\player_detail.csv")
# players_raw.sort_values(by=["player_name", "team_name", "player_role"])

Load team_raw dataset and heroes dataset

In [162]:
team_raw = pd.read_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\raw\team_detail.csv")
team_raw["team_name"] = [i.lower() for i in team_raw["team_name"]]
team_raw = team_raw.replace(["Middle East", "United Arab Emirates"], "MENA")
team_raw = team_raw.replace(["Argentina", "Peru", "Brazil"], "Latin America")
team_raw = team_raw.replace("beaiktaa esports", "besiktas esports")
team_raw["team_id"] = team_raw.index + 1
team_raw

,team_name,team_region,team_id
0,rrq hoshi,Indonesia,1
1,onic,Indonesia,2
2,s8ul esports,India,3
3,team liquid ph,Philippines,4
4,selangor red giants,Malaysia,5
...,...,...,...
159,ra'ad,MENA,160
160,violance,Turkey,161
161,papara supermassive,Turkey,162
162,bad idea right,Turkey,163


In [163]:
heroes_raw = pd.read_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\raw\hero_detail.csv")
heroes_raw["hero_id"] = heroes_raw.index + 1
heroes_raw

,hero_name,role_name,hero_id
0,Akai,Tank,1
1,Alice,Tank,2
2,Atlas,Tank,3
3,Barats,Tank,4
4,Baxia,Tank,5
...,...,...,...
163,Lolita,Support,164
164,Marcel,Support,165
165,Mathilda,Support,166
166,Minotaur,Support,167


Create a draft pick and divide into picks and bans dataset from game based dataset

In [164]:
draft_pick = game_details[["game_keys", "home_team", "away_team", "home_picks", "away_picks", "home_bans", "away_bans", "home_status", "away_status"]].copy()
home_draft_pick = draft_pick.drop(columns=["away_team", "away_picks", "away_bans", "away_status"])
home_draft_pick = home_draft_pick.rename(columns={
    "home_team": "team_name",
    "home_picks": "team_picks",
    "home_bans": "team_bans",
    "home_status": "team_status"
    })
away_draft_pick = draft_pick.drop(columns=["home_team", "home_picks", "home_bans", "home_status"])
away_draft_pick = away_draft_pick.rename(columns={
    "away_team": "team_name",
    "away_picks": "team_picks",
    "away_bans": "team_bans",
    "away_status": "team_status"
    })

In [165]:
picks_stats = pd.concat([home_draft_pick.drop(columns=["team_bans"]), away_draft_pick.drop(columns=["team_bans"])])
team_picks = []
for i in picks_stats["team_picks"]:
    stat = ast.literal_eval(i)
    team_picks.append(stat)
picks_stats["team_picks"] = team_picks
picks_stats = picks_stats.explode("team_picks").reset_index(names="games_id")
picks_stats["team_status"] = np.where(picks_stats["team_status"] == "win", 1, 0)
picks_stats

,games_id,game_keys,team_name,team_picks,team_status
0,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Cici,1
1,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Joy,1
2,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Pharsa,1
3,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Granger,1
4,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Khaleed,1
...,...,...,...,...,...
33673,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Arlott,1
33674,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Guinevere,1
33675,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Lunox,1
33676,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Claude,1


In [166]:
bans_stats = pd.concat([home_draft_pick.drop(columns=["team_picks"]), away_draft_pick.drop(columns=["team_picks"])])
team_bans = []
for i in bans_stats["team_bans"]:
    stat = ast.literal_eval(i)
    team_bans.append(stat)
bans_stats["team_bans"] = team_bans
bans_stats = bans_stats.explode("team_bans").reset_index(names="games_id")
bans_stats["team_status"] = np.where(bans_stats["team_status"] == "win", 1, 0)
bans_stats

,games_id,game_keys,team_name,team_bans,team_status
0,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Yve,1
1,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Phoveus,1
2,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Harith,1
3,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Irithel,1
4,0,2025-02-28 18:30:00+07:00_MPL Philippines Seas...,onic philippines,Badang,1
...,...,...,...,...,...
33298,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Yve,1
33299,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Zhuxin,1
33300,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Fanny,1
33301,3374,2025-12-23 18:00:00+07:00_Games of the Future ...,aurora gaming ph,Moskov,1


Save and store dataset into a csv file

In [167]:
df_tournaments = match_raw[["tournament_name", "tier_name"]].drop_duplicates(subset="tournament_name")
df_teams = team_raw
df_matches = match_result.drop(columns="team_pair")
df_games = game_details.drop(columns="team_pair")
df_picks = picks_stats
df_bans = bans_stats
df_heroes = heroes_raw

In [168]:
df_tournaments.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\interim\tournaments.csv", index=False)
df_teams.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\interim\teams.csv", index=False)
df_matches.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\interim\matches.csv", index=False)
df_games.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\interim\games.csv", index=False)
df_picks.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\interim\picks.csv", index=False)
df_bans.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\interim\bans.csv", index=False)
df_heroes.to_csv(r"E:\End-to-End-MLOps-MWorld-Champions\data\interim\heroes.csv", index=False)

## Insert Dataframe to Database
Create a database and store dataset values

In [ ]:
conn = sqlite3.connect("mlbb.db")
cursor = conn.cursor()

In [170]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS tournaments(
               tournament_id INTEGER PRIMARY KEY AUTOINCREMENT,
               tournament_name TEXT UNIQUE,
               tier_name TEXT
               ) 
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS teams(
               team_id INTEGER PRIMARY KEY AUTOINCREMENT,
               team_name TEXT UNIQUE,
               team_region TEXT
               )
""")
# update next exploration
# cursor.execute("""
# CREATE TABLE IF NOT EXISTS players(
#                player_id INTEGER PRIMARY KEY AUTOINCREMENT,
#                team_id INTEGER
#                player_name TEXT,
#                player_role TEXT,

#                FOREIGN KEY (team_id) REFERENCES teams(team_id)
#                )
# """)

cursor.execute("""
CREATE TABLE IF NOT EXISTS matches(
               match_id INTEGER PRIMARY KEY AUTOINCREMENT,
               match_keys TEXT,
               tournament_id INTEGER,
               bracket_name TEXT,
               date DATE,
               home_team_id INTEGER,
               home_team TEXT,
               away_team_id INTEGER,
               away_team TEXT,

               FOREIGN KEY (tournament_id) REFERENCES tournaments(tournament_id)
               FOREIGN KEY (home_team_id) REFERENCES teams(team_id)
               FOREIGN KEY (away_team_id) REFERENCES teams(team_id)
               )
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS games( 
               game_id INTEGER PRIMARY KEY AUTOINCREMENT,
               game_keys TEXT,
               match_id INTEGER,
               map_name TEXT,
               team_side TEXT,
               duration INTEGER,
               kills INTEGER,
               assists INTEGER,
               deaths INTEGER,
               towers INTEGER,
               tutle INTEGER,
               lords INTEGER,
               gold_total INTEGER,
               gold_5mnt INTEGER,
               gold_10mnt INTEGER,
               gold_15mnt INTEGER,
               gold_20mnt INTEGER,

               FOREIGN KEY (match_id) REFERENCES matches(match_id)
               )
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS picks(
               pick_id INTEGER PRIMARY KEY AUTOINCREMENT,
               game_id INTEGER,
               team_id INTEGER,
               hero_id INTEGER,
               pick_order INTEGER,
               hero_name TEXT,
               
               FOREIGN KEY (game_id) REFERENCES games(game_id),
               FOREIGN KEY (team_id) REFERENCES teams(team_id),
               FOREIGN KEY (hero_id) REFERENCES heroes(hero_id)
               )
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS bans(
               ban_id INTEGER PRIMARY KEY AUTOINCREMENT,
               game_id INTEGER,
               team_id INTEGER,
               hero_id INTEGER,
               ban_order INTEGER,
               hero_name TEXT,

               FOREIGN KEY (game_id) REFERENCES games(game_id),
               FOREIGN KEY (team_id) REFERENCES teams(team_id),
               FOREIGN KEY (hero_id) REFERENCES heroes(hero_id)
               )
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS heroes(
               hero_id INTEGER PRIMARY KEY AUTOINCREMENT,
               hero_name TEXT
               )
""")

conn.commit()

In [171]:
df_tournaments.to_sql("tournaments", conn, if_exists="replace", index=False)
df_teams.to_sql("teams", conn, if_exists="replace", index=False)
df_matches.to_sql("matches", conn, if_exists="replace", index=False)
df_games.to_sql("games", conn, if_exists="replace", index=False)
df_picks.to_sql("picks", conn, if_exists="replace", index=False)
df_bans.to_sql("bans", conn, if_exists="replace", index=False)
df_heroes.to_sql("heroes", conn, if_exists="replace", index=False)

168